RAG Pipeline from data ingestion to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\DataScience\AI Agents\RAG_application\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\DataScience\AI Agents\RAG_application\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### read all the pdf files in the directory and load them as documents

def process_all_pdfs(directory_path):
    
    all_documents = []
    pdf_directory = Path(directory_path)

    #find all the pdf files in the directory
    pdf_files = list(pdf_directory.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()

            # add source information to the metadata of each document
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error occurred while processing {pdf_file.name}: {e}")
       

    return all_documents

all_pdf_docs = process_all_pdfs("../data")


Found 4 PDF files in the directory.
Processing file: Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf
Loaded 21 pages
Processing file: DAY_1_POWER_BI.pdf
Loaded 19 pages
Processing file: decision tree and ensemble leaning.pdf
Loaded 8 pages
Processing file: DL.pdf
Loaded 10 pages


In [3]:
all_pdf_docs

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Datamites-Certified-Data-Scientist-brochure-V25.4', 'source': '..\\data\\pdf\\Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'file_type': 'pdf'}, page_content='CERTIFIED DATA SCIENTIST\nTABLE OF CONTENTS\nDATAMITES® ACCOLADES                …   2\nWHY DATAMITES®                         …   3\nPROGRAM STRUCTURE                            …   4\nREAL-TIME INTERNSHIP              …   5\nJOB  READY PROGRAM                    …    6\nPROGRAM CURRICULUM                   …    7\nADMISSIONS AND CONTACTS                …    20\nPROGRAM BROCHURE\n©DataMites. All content are in this document is copyrighted, \nReproducing any part of the content requires written permission from DataMites®\nYOUR GOAL IS OUR MISSION\nOur aim is to equip learners with the skills nec

In [4]:
# split the documents into smaller chunks using RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""] 
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Example
    if split_docs:
        print(f"content: {split_docs[2].page_content[:500]}")
        print(f"metadata: {split_docs[2].metadata}")
    return split_docs


In [5]:
chunks = split_documents(all_pdf_docs)
chunks

Split 58 documents into 41 chunks.
content: 3
1. Flexible Learning
Learners can repeat sessions, change batches , 
change learning modes, ad-hoc doubts 
sessions anytime. 
2. Job-oriented curriculum
The course curriculum is aligned with Industry 
requirement by expert content team, ensuring 
job-oriented curriculum
3. Elite instructors
Elite mentors and faculties members holding 
real-time experience from leading companies. 
and rom league institutes such as IIMs
4. Exclusive Practice Lab
Learners get exclusive access to AI and Data 
Scie
metadata: {'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Datamites-Certified-Data-Scientist-brochure-V25.4', 'source': '..\\data\\pdf\\Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'total_pages': 21, 'page': 2, 'page_label': '3', 'source_file': 'Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Datamites-Certified-Data-Scientist-brochure-V25.4', 'source': '..\\data\\pdf\\Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'file_type': 'pdf'}, page_content='CERTIFIED DATA SCIENTIST\nTABLE OF CONTENTS\nDATAMITES® ACCOLADES                …   2\nWHY DATAMITES®                         …   3\nPROGRAM STRUCTURE                            …   4\nREAL-TIME INTERNSHIP              …   5\nJOB  READY PROGRAM                    …    6\nPROGRAM CURRICULUM                   …    7\nADMISSIONS AND CONTACTS                …    20\nPROGRAM BROCHURE\n©DataMites. All content are in this document is copyrighted, \nReproducing any part of the content requires written permission from DataMites®\nYOUR GOAL IS OUR MISSION\nOur aim is to equip learners with the skills nec

Embeddings and VectorDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:

    """ Initialize the EmbeddingManager with a specified sentence transformer model.
    
    Args:
        model_name: Huggingface model for the sentence embeddings.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):

        """load the sentence transformer model for generating embeddings"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully.Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """Generate embeddings for a list of texts.
        
        Args:
            texts: List of strings to generate embeddings for.

        Returns:
            A numpy array of shape (len(texts), embedding_dimension) containing the embeddings.
        """

        if self.model is None:
            raise ValueError("Model not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print("Embeddings generated successfully with shape", {embeddings.shape})
            return embeddings
        
        except Exception as e:
            print(f"Error generating embeddings: {e}")

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12161.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully.Embedding dimension: 384


In [8]:
### Vector Store

class VectorStore:

    """ Manages the embeddings in the chroma vector database."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        """ intialize the vector store with chroma client and collection.
        
        Args: collection_name: Name of the collection in chroma to store the embeddings.
              persist_directory: Directory to persist the chroma database.
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        """"Initialize the chroma client and collection for storing embeddings."""

        try:
            """ creating chroma client with persistence settings """
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"})
            print(f"Vector store initialized with collection:{self.collection_name}")
            print(f"Documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):

        """Add documents and their corresponding embeddings to the vector store.
        
        Args:
            documents: List of documents to be stored.
            embeddings: corresponding embeddings of the documents.
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same.")
        print(f"Adding {len(documents)} documents to the vector store...")
        
        # prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            #generate unique id for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #generate metadata for each document
            metadata = dict(doc.metadata)
            metadata["index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #store the document text
            documents_text.append(doc.page_content)

            #store the embedding as list
            embeddings_list.append(embedding.tolist())

        #add to chroma collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

vector_store = VectorStore()
vector_store       


Vector store initialized with collection:pdf_documents
Documents in collection: 287


In [9]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Datamites-Certified-Data-Scientist-brochure-V25.4', 'source': '..\\data\\pdf\\Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'Datamites-Certified-Data-Scientist-brochure-V25.4 (1) (1).pdf', 'file_type': 'pdf'}, page_content='CERTIFIED DATA SCIENTIST\nTABLE OF CONTENTS\nDATAMITES® ACCOLADES                …   2\nWHY DATAMITES®                         …   3\nPROGRAM STRUCTURE                            …   4\nREAL-TIME INTERNSHIP              …   5\nJOB  READY PROGRAM                    …    6\nPROGRAM CURRICULUM                   …    7\nADMISSIONS AND CONTACTS                …    20\nPROGRAM BROCHURE\n©DataMites. All content are in this document is copyrighted, \nReproducing any part of the content requires written permission from DataMites®\nYOUR GOAL IS OUR MISSION\nOur aim is to equip learners with the skills nec

In [10]:
# get the text content from the chunks to generate embeddings
texts = [doc.page_content for doc in chunks]

#generate embeddings for the texts
embeddings = embedding_manager.generate_embeddings(texts)

# add the documents and their embeddings to the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 41 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.00it/s]


Embeddings generated successfully with shape {(41, 384)}
Adding 41 documents to the vector store...
Successfully added 41 documents to the vector store.
Total documents in collection: 328


In [ ]:
class RAGRetriever:

    """handles retrieval of relevant documents based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        """ Initialize the RAGRetriever with a vector store and embedding manager.
        
        Args:
            vector store containing document embeddings and metadata.
            embedding manager for generating query embeddings.
        
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        """ Retrieve relevant documents based on the query.
        
        args:
            query: The input query string for which relevant documents need to be retrieved.
            top_k: The number of top relevant documents to retrieve.
            score_threshold: Minimum cosine similarity score threshold.
        """

        print(f"Retrieving documents for query: {query}")
        print(f"Top_k: {top_k}, Score Threshold: {score_threshold}")

        # generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search the vector store for similar documents
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents after filtering")
            
            else:
                print(f"no document found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise   

rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [12]:
rag_retriever.retrieve("whyPower BI")

Retrieving documents for query: whyPower BI
Top_k: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.77it/s]

Embeddings generated successfully with shape {(1, 384)}
Retrieved 5 documents after filtering


[{'id': 'doc_e99d5d34_31',
  'content': '"Empowering Decisions: The Power of\nPower BI"\nPower BI\n`',
  'metadata': {'page': 0,
   'index': 31,
   'file_type': 'pdf',
   'total_pages': 19,
   'source': '..\\data\\pdf\\DAY_1_POWER_BI.pdf',
   'producer': 'Canva',
   'keywords': 'DAF-dWpjHwU,BAEHNqb2rAo',
   'title': 'DAY_1_POWER_BI',
   'creator': 'Canva',
   'author': 'Mohammed Luqman',
   'page_label': '1',
   'creationdate': '2024-05-04T07:56:01+00:00',
   'source_file': 'DAY_1_POWER_BI.pdf',
   'content_length': 56,
   'moddate': '2024-05-04T07:55:58+00:00'},
  'similarity_score': 0.29234403371810913,
  'distance': 0.7076559662818909,
  'rank': 1},
 {'id': 'doc_b2c96aa6_31',
  'content': '"Empowering Decisions: The Power of\nPower BI"\nPower BI\n`',
  'metadata': {'title': 'DAY_1_POWER_BI',
   'creationdate': '2024-05-04T07:56:01+00:00',
   'page': 0,
   'author': 'Mohammed Luqman',
   'content_length': 56,
   'producer': 'Canva',
   'keywords': 'DAF-dWpjHwU,BAEHNqb2rAo',
   'sourc

In [13]:
rag_retriever.retrieve("what exactly is Business Intelligence,anyway?")

Retrieving documents for query: what exactly is Business Intelligence,anyway?
Top_k: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.92it/s]

Embeddings generated successfully with shape {(1, 384)}
Retrieved 5 documents after filtering


[{'id': 'doc_7add5829_32',
  'content': 'what exactly is Business Intelligence,\nanyway?\nbusiness intelligence is all about leveraging data to make better\ndecisions. This can take many forms and is not necessarily restricted to\njust business. We use data in our personal lives to make\nbetter decisions as well,\nFor example, if we are remodeling a bathroom, we get multiple\nquotes from different firms. The prices and details in these quotes are\npieces of data that allow us to make an informed decision in terms of\nwhich company to choose',
  'metadata': {'moddate': '2024-05-04T07:55:58+00:00',
   'source_file': 'DAY_1_POWER_BI.pdf',
   'creator': 'Canva',
   'content_length': 496,
   'total_pages': 19,
   'page': 1,
   'file_type': 'pdf',
   'index': 32,
   'creationdate': '2024-05-04T07:56:01+00:00',
   'source': '..\\data\\pdf\\DAY_1_POWER_BI.pdf',
   'keywords': 'DAF-dWpjHwU,BAEHNqb2rAo',
   'author': 'Mohammed Luqman',
   'title': 'DAY_1_POWER_BI',
   'page_label': '2',
   'prod

Intergrating VectorDB Context Pipeline with LLM

In [15]:
# Simple rag pipeline with groq llm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# initialize the groq llm(set up groq api key in environment)
groq_llm_api_key = os.getenv("GROQ_API_KEY") 

llm = ChatGroq(groq_api_key = groq_llm_api_key, model="openai/gpt-oss-120b",temperature=0.1, max_tokens=1024)

# rag function: retrieve context + generate answer using llm
def chat_rag(query, rag_retriever, llm, top_k=3):

    #retrieve the relavant context
    results = rag_retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "Sorry, I couldn't find any relevant information to answer your question."
    
    #generate answer using the gwoq llm
    prompt = f""" use the following context to answer the question concisely.
            If you don't know the answer, say you don't know. 
            
            Context: {context} 

            Question: {query}

            Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [16]:
answer = chat_rag("what exactly is Business Intelligence,anyway?", rag_retriever, llm)
print(answer)

Retrieving documents for query: what exactly is Business Intelligence,anyway?
Top_k: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.61it/s]

Embeddings generated successfully with shape {(1, 384)}
Retrieved 3 documents after filtering


Business Intelligence (BI) is the practice of collecting, analyzing, and using data to make more informed decisions. It isn’t limited to companies—any situation where you gather data (like comparing multiple quotes for a bathroom remodel) and use that information to choose the best option is an example of BI.


In [17]:
# rag with advanced features

def advanced_chat_rag(query, rag_retriever, llm, top_k=3, min_score=0.2, return_context=False):

    '''
    Advanced RAG function with additional features.
    returns answer, sources, confidence score and optionally the retrieved context.
    '''
    results = rag_retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {"answer": "Sorry, no relevant information found.", "sources": [], "confidence": 0.0, "context": ""} if return_context else {"answer": "Sorry, no relevant information found.", "source": [], "confidence": 0.0}

    #prepare the context and sources for the answer generation
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('score', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'similarity_score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'  
    }for doc in results]

    confidence = max(doc['similarity_score'] for doc in results)
 
    #generate answer using the groq llm
    prompt = f""" use the following context to answer the question concisely.
            If you don't know the answer, say you don't know. 
            
            Context: {context} 

            Question: {query}

            Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

#example usage of the advanced rag function
response = advanced_chat_rag("what exactly is Business Intelligence,anyway?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", response['answer'])
print("Sources:", response['sources'])
print("Confidence Score:", response['confidence'])
print("Context Preview:", response['context'][:300])

Retrieving documents for query: what exactly is Business Intelligence,anyway?
Top_k: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.00it/s]

Embeddings generated successfully with shape {(1, 384)}
Retrieved 3 documents after filtering


Answer: Business Intelligence is the practice of gathering, analyzing, and using data to make more informed decisions. It isn’t limited to companies—any situation where you compare data (like multiple quotes for a bathroom remodel) to choose the best option is an example of Business Intelligence in action.
Sources: [{'source': 'DAY_1_POWER_BI.pdf', 'page': 1, 'similarity_score': 0.6421187520027161, 'preview': 'what exactly is Business Intelligence,\nanyway?\nbusiness intelligence is all about leveraging data to make better\ndecisions. This can take many forms and is not necessarily restricted to\njust business. We use data in our personal lives to make\nbetter decisions as well,\nFor example, if we are remodeli...'}, {'source': 'DAY_1_POWER_BI.pdf', 'page': 1, 'similarity_score': 0.6421187520027161, 'preview': 'what exactly is Business Intelligence,\nanyway?\nbusiness intelligence is all about leveraging data to make better\ndecisions. This can take many forms and is not necessarily re

In [18]:
"""Advanced RAG pipeline with streaming, citations, history, and summary."""

from typing import List, Dict, Any, Optional


class AdvancedRag:

    def __init__(self, rag_retriever, llm):
        self.retriever = rag_retriever
        self.llm = llm
        self.history: List[Dict[str, Any]] = []

    def _deduplicate(self, docs: List[Dict]) -> List[Dict]:
        """Remove duplicate documents based on (source, page) identity."""
        seen = set()
        unique = []
        for doc in docs:
            key = (
                doc["metadata"].get("source_file", ""),
                doc["metadata"].get("page", ""),
            )
            if key not in seen:
                seen.add(key)
                unique.append(doc)
        return unique

    def query(
        self,
        question: str,
        top_k: int = 3,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, Any]:
        """
        Advanced RAG query with streaming, citations, history, and summary.

        Args:
            question:   The user's question.
            top_k:      Maximum number of documents to retrieve.
            min_score:  Minimum similarity score threshold.
            stream:     If True, stream the LLM answer token-by-token to stdout.
            summarize:  If True, append a 2-sentence summary to the result.

        Returns:
            A dict with keys: question, answer, sources, summary, history.
        """
        #Retrieve
        retrieved_docs = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score
        )
        retrieved_docs = self._deduplicate(retrieved_docs)

        # Build context & sources 
        if not retrieved_docs:
            answer = "No relevant information found."
            sources: List[Dict] = []
        else:
            context = "\n\n".join(doc["content"] for doc in retrieved_docs)

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("score", "unknown"),
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "...",
                }
                for doc in retrieved_docs
            ]

            prompt = (
                "Use the following context to answer the question concisely.\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Answer:"
            )

            #Streaming
            if stream:
                # Stream the *answer* token-by-token via the LLM's streaming API.
                print("Generating answer (streaming):")
                answer_chunks: List[str] = []
                for chunk in self.llm.stream([prompt]):
                    token = chunk.content if hasattr(chunk, "content") else str(chunk)
                    print(token, end="", flush=True)
                    answer_chunks.append(token)
                print() 
                answer = "".join(answer_chunks)
            else:
                llm_response = self.llm.invoke([prompt])
                answer = llm_response.content

        #Attach inline citations
        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations
            else answer
        )

        #Optional summary 
        summary: Optional[str] = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_response = self.llm.invoke([summary_prompt])
            summary = summary_response.content

        #store history 
        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }


#Example usage
if __name__ == "__main__":
    advanced_rag = AdvancedRag(rag_retriever, llm)
    response = advanced_rag.query(
        "What exactly is Business Intelligence, anyway?",
        top_k=3,
        min_score=0.1,
        stream=True,
        summarize=True,
    )
    print("\nFinal answer:\n", response["answer"])
    print("\nSummary:\n", response["summary"])
    print("\nHistory (last entry):\n", response["history"][-1])

Retrieving documents for query: What exactly is Business Intelligence, anyway?
Top_k: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.62it/s]

Embeddings generated successfully with shape {(1, 384)}
Retrieved 3 documents after filtering
Generating answer (streaming):


Business Intelligence (BI) is the practice of collecting, integrating, analyzing, and visualizing data so that individuals and organizations can make informed, data‑driven decisions. It turns raw data—such as sales figures, customer interactions, or even personal quotes for a home remodel—into actionable insights, helping users choose the best options and improve outcomes.

Final answer:
 Business Intelligence (BI) is the practice of collecting, integrating, analyzing, and visualizing data so that individuals and organizations can make informed, data‑driven decisions. It turns raw data—such as sales figures, customer interactions, or even personal quotes for a home remodel—into actionable insights, helping users choose the best options and improve outcomes.

Citations:
[1] DAY_1_POWER_BI.pdf (page 1)

Summary:
 Business Intelligence (BI) involves gathering, integrating, analyzing, and visualizing data to enable data‑driven decision‑making. By converting raw information—like sales numbe